In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

# Create project directory
import os
PROJECT_ROOT = '/gdrive/MyDrive/CapRL_Comparison'
os.makedirs(PROJECT_ROOT, exist_ok=True)




Mounted at /gdrive


In [2]:
import json
import os
import torch
import re
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

print("[1/6] Installing dependencies...")
os.system("pip install -q transformers bitsandbytes datasets pillow -q")

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


[1/6] Installing dependencies...


In [5]:
import sys
PROJECT_ROOT = "/gdrive/MyDrive/CapRL_Comparison"
SHAREGPT_JSON = f"{PROJECT_ROOT}/datasets/sharegpt_training_5k.json"
COCO_ROOT = "/gdrive/MyDrive/coco2017"
OUTPUT_JSONL = f"/gdrive/MyDrive/CapRL_Project/caprl_mcq_dataset_final.jsonl"

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
MAX_NEW_TOKENS = 500
BATCH_SIZE = 1  # Process one caption at a time for quality
MCQS_PER_IMAGE = 4
TEMPERATURE = 0.7
TOP_P = 0

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_JSONL), exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Input JSON: {SHAREGPT_JSON}")
print(f"COCO root: {COCO_ROOT}")
print(f"Output file: {OUTPUT_JSONL}")

Project root: /gdrive/MyDrive/CapRL_Comparison
Input JSON: /gdrive/MyDrive/CapRL_Comparison/datasets/sharegpt_training_5k.json
COCO root: /gdrive/MyDrive/coco2017
Output file: /gdrive/MyDrive/CapRL_Project/caprl_mcq_dataset.jsonl


# Loading Sharegpt data

### Debugging: Generate MCQs for a Single Record

This cell will process only the first record from `records`, generate MCQs for it, and attempt to save them to the `OUTPUT_JSONL` file. This helps isolate any issues with the saving process.

In [ ]:
print("\n[DEBUG] Generating MCQs for a single record...")

debug_record = records[0]  # Get the first record
debug_image_id = debug_record["image_id"]
debug_caption = debug_record["caption"]

debug_mcqs = generate_mcqs_for_caption(debug_caption, debug_image_id, MCQS_PER_IMAGE)

if debug_mcqs:
    print(f"Generated {len(debug_mcqs)} MCQs for image: {debug_image_id}")
    with open(OUTPUT_JSONL, "a", encoding="utf-8") as outf:
        for mcq in debug_mcqs:
            mcq["image_path"] = debug_image_id
            outf.write(json.dumps(mcq, ensure_ascii=False) + "\n")
    print(f"Successfully wrote {len(debug_mcqs)} MCQs to {OUTPUT_JSONL}")
else:
    print(f"No MCQs generated for image: {debug_image_id}")

# Verify by reading the first few lines of the output file
print("\n[DEBUG] Verifying contents of output file:")
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:  # Read up to 5 lines for verification
            break
        print(line.strip())


In [ ]:

# ============================================================================
# STEP 1: LOAD SHAREGPT DATA
# ============================================================================
print("\n[2/6] Loading ShareGPT data... ")

if not os.path.exists(SHAREGPT_JSON):
    raise FileNotFoundError(f"ShareGPT JSON not found: {SHAREGPT_JSON}")

with open(SHAREGPT_JSON, "r", encoding="utf-8") as f:
    sharegpt_data = json.load(f)

print(f"Loaded {len(sharegpt_data)} samples from ShareGPT")

# Extract image paths and captions
records = []
for sample in sharegpt_data:
    image_id = sample.get("image_id", "")

    if not image_id:
        continue

    # Extract caption from conversations
    conversations = sample.get("conversations", [])
    caption = None

    for turn in conversations:
        if turn.get("from") == "gpt":
            caption = turn.get("value", "").strip()
            break

    if not caption or len(caption) < 20:
        continue

    records.append({
        "image_id": image_id,
        "caption": caption
    })

print(f"Valid records: {len(records)}")




[2/6] Loading ShareGPT data... 
Loaded 5000 samples from ShareGPT
Valid records: 5000


# STEP 2: LOAD MISTRAL-7B WITH 4-BIT QUANTIZATION

In [ ]:
print("\n[3/6] Loading Mistral-7B-Instruct (4-bit quantized)...")

# 4-bit quantization config to fit in Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()
print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")




[3/6] Loading Mistral-7B-Instruct (4-bit quantized)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.1
Model dtype: torch.float16


# STEP 3: MCQ GENERATION SYSTEM PROMPT

In [ ]:

# ============================================================================
# STEP 3: MCQ GENERATION SYSTEM PROMPT
# ============================================================================
SYSTEM_PROMPT = """You are an expert at creating challenging, knowledge-based multiple choice questions about image content.

GUIDELINES:
- Generate questions that REQUIRE understanding BOTH the visual content AND the detailed caption
- Create open-ended questions that test deep comprehension
- Focus on relationships, spatial arrangements, actions, context, composition
- AVOID trivial questions: colors, simple counts, single object names, obvious attributes
- Make distractors plausible but clearly wrong when you understand the image
- Ensure exactly ONE correct answer

OUTPUT FORMAT:
Return ONLY a valid JSON object (no markdown, no extra text) with this structure:
{
  "mcqs": [
    {
      "question": "Full question text",
      "options": {
        "A": "Option A text",
        "B": "Option B text",
        "C": "Option C text",
        "D": "Option D text"
      },
      "correct": "B"
    }
  ]
}"""


# STEP 4: MCQ GENERATION FUNCTION

In [ ]:
def generate_mcqs_for_caption(caption, image_id, num_mcqs=4):
    """
    Generate MCQs for a single caption using Mistral-7B.
    Returns: list of MCQ dicts with format matching output requirements
    """

    # Combine SYSTEM_PROMPT with the user request to ensure JSON output
    user_prompt = f"""{SYSTEM_PROMPT}

Based on this detailed image caption, generate {num_mcqs} challenging multiple choice questions that require understanding the image content and caption details.

Caption:
{caption}
"""

    # Format for Mistral Chat
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    # Tokenize and generate
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # Only decode the NEW tokens (exclude the prompt)
        input_length = inputs.shape[1]
        generated_tokens = outputs[0][input_length:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Extract JSON from response
        try:
            # Clean up markdown code blocks if present
            clean_response = response.replace("```json", "").replace("```", "").strip()

            # Find JSON in response
            start_idx = clean_response.find('{')
            end_idx = clean_response.rfind('}') + 1

            if start_idx == -1 or end_idx <= start_idx:
                return []

            json_str = clean_response[start_idx:end_idx]
            parsed = json.loads(json_str)

            mcqs_raw = parsed.get("mcqs", [])

            # Convert to required format
            mcqs_formatted = []
            for idx, mcq in enumerate(mcqs_raw):
                options = mcq.get("options", {})
                question_text = mcq.get("question", "")
                correct = mcq.get("correct", "A")

                # Format question with options
                formatted_q = f"{question_text}\n   - A) {options.get('A', '')}\n   - B) {options.get('B', '')}\n   - C) {options.get('C', '')}\n   - D) {options.get('D', '')}"

                mcqs_formatted.append({
                    "id": f"caprl_{image_id}_q{idx}",
                    "image_id": image_id,
                    "question": formatted_q,
                    "correct_label": correct
                })

            return mcqs_formatted[:num_mcqs]  # Limit to requested number

        except json.JSONDecodeError:
            return []

    except Exception as e:
        print(f"Error generating MCQs for {image_id}: {str(e)}")
        return []



# STEP 5: BATCH MCQ GENERATION

In [ ]:
print("\n[4/6] Generating MCQs... ")
print(f"Will generate {MCQS_PER_IMAGE} MCQs per image")
print(f"Total MCQs to generate: ~{len(records) * MCQS_PER_IMAGE}")

all_mcqs = []
skipped = 0
generated = 0

records_to_process = list(reversed(records))[1600:2600]

with open(OUTPUT_JSONL, "w", encoding="utf-8") as outf:
    for idx, record in enumerate(tqdm(records_to_process, desc="Generating MCQs")):


        image_id = record["image_id"]
        image_path = image_id
        caption = record["caption"]

        # Generate MCQs
        mcqs = generate_mcqs_for_caption(caption, image_id, MCQS_PER_IMAGE)
        if idx == 1:
          print(mcqs)
        if not mcqs:
            skipped += 1
            continue

        # Add image_path to each MCQ
        for mcq in mcqs:
            mcq["image_path"] = image_path
            outf.write(json.dumps(mcq, ensure_ascii=False) + "\n")

            generated += 1
        if idx % 10 == 0:
            outf.flush()
        all_mcqs.extend(mcqs)

print(f"\n✓ Generated: {generated} MCQs")
print(f"✗ Skipped: {skipped} images")
print(f"Total MCQs saved: {len(all_mcqs)}")


[4/6] Generating MCQs... 
Will generate 4 MCQs per image
Total MCQs to generate: ~20000


Generating MCQs:   0%|          | 2/1000 [01:00<8:27:28, 30.51s/it]

[{'id': 'caprl_coco/train2017/000000016955.jpg_q0', 'image_id': 'coco/train2017/000000016955.jpg', 'question': 'What is the time displayed on the clock in the image?\n   - A) 3:45\n   - B) 4:15\n   - C) 5:00\n   - D) 6:00', 'correct_label': 'B'}, {'id': 'caprl_coco/train2017/000000016955.jpg_q1', 'image_id': 'coco/train2017/000000016955.jpg', 'question': 'What is the purpose of the moving walkway in the image?\n   - A) To transport passengers to their gates\n   - B) To provide a place for passengers to rest\n   - C) To display advertisements\n   - D) To serve as a decorative element', 'correct_label': 'A'}, {'id': 'caprl_coco/train2017/000000016955.jpg_q2', 'image_id': 'coco/train2017/000000016955.jpg', 'question': 'What are the signs in the image indicating?\n   - A) The location of the service desk\n   - B) The direction for passengers to proceed\n   - C) The location of the coffee bar\n   - D) The location of the baggage claim', 'correct_label': 'B'}, {'id': 'caprl_coco/train2017/00

Generating MCQs:  43%|████▎     | 428/1000 [3:29:43<4:33:01, 28.64s/it]

### Inspecting `OUTPUT_JSONL` Contents

This cell reads and prints the first 5 lines of the `OUTPUT_JSONL` file to help you verify its content.

In [8]:
import json
import os

if os.path.exists(OUTPUT_JSONL):
    print(f"Reading contents from: {OUTPUT_JSONL}")
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        print("\n--- First 5 lines of OUTPUT_JSONL ---")
        for i, line in enumerate(f):
            if i >= 5:  # Print up to 5 lines
                break
            try:
                # Attempt to parse as JSON for better readability
                parsed_line = json.loads(line)
                print(json.dumps(parsed_line, indent=2, ensure_ascii=False))
            except json.JSONDecodeError:
                print(line.strip()) # Print raw line if not valid JSON
        print("--------------------------------------")
else:
    print(f"The file {OUTPUT_JSONL} does not exist yet.")

Reading contents from: /gdrive/MyDrive/CapRL_Project/caprl_mcq_dataset_final.jsonl

--- First 5 lines of OUTPUT_JSONL ---
{
  "id": "caprl_coco/train2017/000000015175.jpg_q0",
  "image_id": "coco/train2017/000000015175.jpg",
  "question": "What is the man doing in the image?\n   - A) He is standing still on the surfboard\n   - B) He is skillfully riding a wave on a blue surfboard\n   - C) He is swimming in the ocean\n   - D) He is jumping off the surfboard",
  "correct_label": "B",
  "image_path": "coco/train2017/000000015175.jpg"
}
{
  "id": "caprl_coco/train2017/000000015175.jpg_q1",
  "image_id": "coco/train2017/000000015175.jpg",
  "question": "What is the color of the man's wetsuit?\n   - A) Red\n   - B) Black\n   - C) Yellow\n   - D) Green",
  "correct_label": "B",
  "image_path": "coco/train2017/000000015175.jpg"
}
{
  "id": "caprl_coco/train2017/000000015175.jpg_q2",
  "image_id": "coco/train2017/000000015175.jpg",
  "question": "What is the shape of the wave that the man is ri

In [14]:

import json
import os

if os.path.exists(OUTPUT_JSONL):
    print(f"Reading contents from: {OUTPUT_JSONL}")
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        print("\n--- First 5 lines of OUTPUT_JSONL ---")
        for i, line in enumerate(f):
            if i >= 5:  # Print up to 5 lines
                pass
            elif i == 10:
                break
            try:
                # Attempt to parse as JSON for better readability
                parsed_line = json.loads(line)
                print(json.dumps(parsed_line, indent=2, ensure_ascii=False))
                sample = parsed_line
                print(f"ID: {sample['id']}")
                print(f"Image: {sample['image_path']}")
                print(f"Question:\n{sample['question']}")
                print(f"Correct: {sample['correct_label']}")
            except json.JSONDecodeError:
                print(line.strip()) # Print raw line if not valid JSON
        print("--------------------------------------")
else:
    print(f"The file {OUTPUT_JSONL} does not exist yet.")





Streaming output truncated to the last 5000 lines.
Question:
What is the mood of the scene?
   - A) Calm
   - B) Lively
   - C) Serious
   - D) Dull
Correct: B
{
  "id": "caprl_coco/train2017/000000021510.jpg_q0",
  "image_id": "coco/train2017/000000021510.jpg",
  "question": "What is the position of the black pony in the image?\n   - A) On the right side of the image\n   - B) On the left side of the image\n   - C) In the middle of the image\n   - D) Facing upwards",
  "correct_label": "B",
  "image_path": "coco/train2017/000000021510.jpg"
}
ID: caprl_coco/train2017/000000021510.jpg_q0
Image: coco/train2017/000000021510.jpg
Question:
What is the position of the black pony in the image?
   - A) On the right side of the image
   - B) On the left side of the image
   - C) In the middle of the image
   - D) Facing upwards
Correct: B
{
  "id": "caprl_coco/train2017/000000021510.jpg_q1",
  "image_id": "coco/train2017/000000021510.jpg",
  "question": "What is the color of the tall church spir

# STEP 6: VALIDATION & SUMMARY

In [9]:
print("\n[5/6] Validating output...")

# Load and validate
validation_count = 0
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        try:
            mcq = json.loads(line)
            assert "id" in mcq
            assert "image_path" in mcq
            assert "question" in mcq
            assert "correct_label" in mcq
            validation_count += 1
        except Exception as e:
            print(f"Error validating line: {line.strip()} - {e}")
            pass

print(f"✓ Validated {validation_count} MCQs in output file")

# # ============================================================================
# # SUMMARY
# # ============================================================================
# print("\n[6/6] Summary")
# print("="*70)
# print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# print(f"Input images: {len(records)}")
# print(f"Total MCQs generated: {generated}")
# print(f"Average MCQs per image: {generated / len(records):.2f}" if records else "N/A")
# print(f"Output file: {OUTPUT_JSONL}")
# print(f"File size: {os.path.getsize(OUTPUT_JSONL) / 1024 / 1024:.2f} MB")
# print("="*70)

# Show sample MCQ
print("\nSample MCQ:")
if all_mcqs:
    sample = all_mcqs[10]
    print(f"ID: {sample['id']}")
    print(f"Image: {sample['image_path']}")
    print(f"Question:\n{sample['question']}")
    print(f"Correct: {sample['correct_label']}")

print("\n✅ MCQ generation complete!  Ready for RL training.")
print(f"Load the dataset with: dataset = load_dataset('json', data_files='{OUTPUT_JSONL}')")


[5/6] Validating output...
✓ Validated 7944 MCQs in output file

Sample MCQ:


NameError: name 'all_mcqs' is not defined

Give me the versions of all the libraries that i will need
